# CosmoLang Native Foundation Model Training (LoRA SFT)
Fine-tunes **Qwen2.5-Coder-1.5B-Instruct** into a specialized inductive neuro-symbolic synthesizer for ARC-AGI-3.

**Architecture:** Autoregressive Spatial Induction with Closed-Algebra CosmoLang AST + LACA Zero-Error Guard.

In [ ]:
# Step 1: Install Required Libraries
!pip install -q transformers peft trl datasets accelerate bitsandbytes

In [ ]:
# Step 2: GPU Verification
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))


In [ ]:
# Step 3: Embed CosmoLang Native Engine
import os, sys
!git clone https://github.com/OCBFUGA/CosmoLang.git /kaggle/working/CosmoLang
sys.path.insert(0, '/kaggle/working/CosmoLang')
from cosmolang.native_model import CosmoGrammar, CosmoProgram, CosmoVerifier, SyntheticTaskGenerator
print('CosmoLang Native Modules successfully loaded!')

In [ ]:
# Step 4: Model & LoRA Configuration
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
print(f'Loading {MODEL_ID}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Training Execution with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import os, torch

DATASET_PATH = 'cosmolang_train_10k.jsonl'
if not os.path.exists(DATASET_PATH):
    print('Generating 10,000 synthetic training tasks...')
    from cosmolang.native_model.export_dataset import export_synthetic_dataset
    export_synthetic_dataset(output_file=DATASET_PATH, total_samples=10000, max_depth=3)

raw_dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
print('Raw dataset loaded, size:', len(raw_dataset))

# Format prompts using ChatML template
def format_prompts(batch):
    texts = []
    for msgs in batch['messages']:
        formatted = tokenizer.apply_chat_template(msgs, tokenize=False)
        texts.append(formatted)
    return {'text': texts}

dataset = raw_dataset.map(format_prompts, batched=True)
print('Dataset formatted with ChatML successfully!')

training_args = TrainingArguments(
    output_dir='./cosmolang_native_checkpoints',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy='epoch',
    bf16=True if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else False,
    fp16=True if torch.cuda.is_available() and not torch.cuda.is_bf16_supported() else False,
    report_to='none'
)

# SFTTrainer: model is already a PeftModel, so do not pass peft_config again
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args
)

print('Starting LoRA SFT Training on GPU...')
trainer.train()

print('Saving fine-tuned adapter to ./cosmolang_native_adapter...')
model.save_pretrained('./cosmolang_native_adapter')
tokenizer.save_pretrained('./cosmolang_native_adapter')
print('TRAINING COMPLETED SUCCESSFULLY! Adapter saved.')
